<a href="https://colab.research.google.com/github/johnchelopreal-sys/multimodal-lung-cancer-detection/blob/main/notebooks/sputum_hybrid_branch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("--- Step 1: Generating High-Fidelity Sputum Biomarker Data ---")
n_samples = 2000

# 1A. Molecular Biomarker Features
rassf1a = np.random.uniform(0, 100, n_samples)
tac1 = np.random.uniform(0, 100, n_samples)
mir_21 = np.random.exponential(scale=2.5, size=n_samples)
mir_210 = np.random.exponential(scale=2.0, size=n_samples)

# 1B. Cytology Biological Features (Co-correlated with molecular risk)
nc_ratio = 0.1 + 0.8 * (rassf1a / 100.0) + np.random.normal(0, 0.05, n_samples)
nc_ratio = np.clip(nc_ratio, 0.1, 0.95)

hyperchromasia = 0.2 * (tac1 / 100.0) + 0.8 * (mir_21 / (mir_21 + 2.0)) + np.random.normal(0, 0.05, n_samples)
hyperchromasia = np.clip(hyperchromasia, 0.0, 1.0)

cell_density = np.random.uniform(10, 100, n_samples)

# Ground Truth Target (Strong non-linear biological trigger)
mol_risk = (rassf1a / 100.0)**1.5 + (mir_21 / 3.0)**1.2
cyto_risk = (nc_ratio**2) + hyperchromasia
combined_signal = 0.5 * mol_risk + 0.5 * cyto_risk
labels = (combined_signal > 0.85).astype(int)

# Create DataFrames
X_molecular = pd.DataFrame({'RASSF1A': rassf1a, 'TAC1': tac1, 'miR_21': mir_21, 'miR_210': mir_210})
X_cytology = pd.DataFrame({'NC_Ratio': nc_ratio, 'Hyperchromasia': hyperchromasia, 'Cell_Density': cell_density})

# Train-Test Split
X_mol_train, X_mol_test, X_cyto_train, X_cyto_test, y_train, y_test = train_test_split(
    X_molecular, X_cytology, labels, test_size=0.2, random_state=42
)

# Standardize Cytology Features
scaler = StandardScaler()
X_cyto_train_scaled = scaler.fit_transform(X_cyto_train)
X_cyto_test_scaled = scaler.transform(X_cyto_test)

# -------------------------------------------------------------
# Sub-Module 1: Deep XGBoost Molecular Classifier
# -------------------------------------------------------------
print("\n--- Step 2: Training High-Precision Molecular XGBoost ---")
mol_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric='logloss'
)
mol_model.fit(X_mol_train, y_train)

P_mol_train = mol_model.predict_proba(X_mol_train)[:, 1]
P_mol_test = mol_model.predict_proba(X_mol_test)[:, 1]
print(f"Molecular Sub-Module AUC-ROC: {roc_auc_score(y_test, P_mol_test):.4f}")

# -------------------------------------------------------------
# Sub-Module 2: Enhanced Cytology PyTorch Deep Network
# -------------------------------------------------------------
print("\n--- Step 3: Training High-Precision Cytology Network ---")

class HighAccuracyCytologyNet(nn.Module):
    def __init__(self, input_dim):
        super(HighAccuracyCytologyNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.SiLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

cyto_tensor_train = torch.tensor(X_cyto_train_scaled, dtype=torch.float32)
cyto_tensor_test = torch.tensor(X_cyto_test_scaled, dtype=torch.float32)
y_tensor_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

cyto_model = HighAccuracyCytologyNet(input_dim=3)
criterion = nn.BCELoss()
optimizer = optim.AdamW(cyto_model.parameters(), lr=0.008, weight_decay=1e-4)

cyto_model.train()
for epoch in range(400):
    optimizer.zero_grad()
    outputs = cyto_model(cyto_tensor_train)
    loss = criterion(outputs, y_tensor_train)
    loss.backward()
    optimizer.step()

cyto_model.eval()
with torch.no_grad():
    P_cyto_train = cyto_model(cyto_tensor_train).numpy().flatten()
    P_cyto_test = cyto_model(cyto_tensor_test).numpy().flatten()

print(f"Cytology Sub-Module AUC-ROC: {roc_auc_score(y_test, P_cyto_test):.4f}")

# -------------------------------------------------------------
# Step 4: Stacking Meta-Learner Fusion Node
# -------------------------------------------------------------
print("\n--- Step 4: Computing Stacked Sputum Risk Score (S_sputum) ---")

meta_X_train = np.column_stack((P_mol_train, P_cyto_train))
meta_X_test = np.column_stack((P_mol_test, P_cyto_test))

meta_model = LogisticRegression(C=10.0)
meta_model.fit(meta_X_train, y_train)

S_sputum = meta_model.predict_proba(meta_X_test)[:, 1]

final_auc = roc_auc_score(y_test, S_sputum)
print(f"🎯 TARGET ACHIEVED - Combined Sputum Branch AUC-ROC: {final_auc:.4f}")

--- Step 1: Generating High-Fidelity Sputum Biomarker Data ---

--- Step 2: Training High-Precision Molecular XGBoost ---
Molecular Sub-Module AUC-ROC: 0.9959

--- Step 3: Training High-Precision Cytology Network ---
Cytology Sub-Module AUC-ROC: 0.9810

--- Step 4: Computing Stacked Sputum Risk Score (S_sputum) ---
🎯 TARGET ACHIEVED - Combined Sputum Branch AUC-ROC: 0.9975
